In [ ]:
import csv
import pandas as pd
import re

In [ ]:
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col

# Recodage variable q25_statut

On recode les personnes qui ont répondu Autre à la question : "Quel est votre statut ?"
Je propose également de faire une autre variable : "Doctorant/non doctorant"

In [ ]:


df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"chercheur")), "q25_statut_rec"] = "Chercheur"
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"doctorante|docteur")), "q25_statut_rec"] = "Doctorant"
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"ing")), "q25_statut_rec"] = "Personnel d'appui à la recherche"

df0.loc[df0.q25_statut=="Doctorant·e", "q25_statut_rec"] = "Doctorant"
df0.loc[df0.q25_statut=="Enseignant·e chercheur·e", "q25_statut_rec"] = "Chercheur"
df0.loc[df0.q25_statut=="Personnel de soutien à la recherche", "q25_statut_rec"] = "Personnel d'appui à la recherche"
df0.q25_statut_rec.value_counts()


dic_new_var = {'name':'110. statut_rec', 
               'label':'q25_statut_rec', 
               'group':'12_descript_repondant', 
               'personal_data':False, 
               'type':'simple_nominal', 
               'opened_question':False,
               'type_panda':df0.q25_statut_rec.dtypes,
               'no_question':25,
               'question':df_col.question.loc[df_col.label=="q25_statut"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])

new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)


In [ ]:
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"chercheur")), "q25_doctorant"] = False
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"doctorante|docteur")), "q25_doctorant"] = True
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"ing")), "q25_doctorant"] = False

df0.loc[df0.q25_statut=="Doctorant·e", "q25_doctorant"] = True
df0.loc[df0.q25_statut=="Enseignant·e chercheur·e", "q25_doctorant"] = False
df0.loc[df0.q25_statut=="Personnel de soutien à la recherche", "q25_doctorant"] = False
df0.q25_doctorant.value_counts()

dic_new_var = {'name':'110. doctorant', 
               'label':'q25_doctorant', 
               'group':'12_descript_repondant', 
               'personal_data':False, 
               'type':'simple_nominal', 
               'opened_question':False,
               'type_panda':df0.q25_doctorant.dtypes,
               'no_question':25,
               'question':df_col.question.loc[df_col.label=="q25_statut"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])

new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)


In [ ]:
df0.loc[df0.q3_octavi_depot.str.contains("Non|Je ne connais pas"), "q3_octavi_rec"] = "Non"
df0.loc[~df0.q3_octavi_depot.str.contains("Non|Je ne connais pas"), "q3_octavi_rec"] = "Oui"


In [ ]:
dic_new_var = {'name':'3. octavi_depot_rec',
               'label':'q3_octavi_rec',
               'group':'2_bib_num',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df0.q3_octavi_rec.dtypes,
               'no_question':3,
               'question':df_col.question.loc[df_col.label=="q3_octavi_depot"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

In [ ]:
df_col.to_csv("../le_questionnaire/dico_variable.csv", sep=",", index= False)